In [0]:
from pyspark.sql import functions as F

staging_order_items = spark.table("brazilian_ecommerce.prod.staging_order_items")
staging_sellers = spark.table("brazilian_ecommerce.prod.staging_sellers")
staging_orders = spark.table("brazilian_ecommerce.prod.staging_orders")

# --- identify orders with exactly one seller ---
single_seller_orders = (
    staging_order_items
    .groupBy("order_id")
    .agg(
        F.countDistinct("seller_id").alias("seller_count"),
        F.first("seller_id").alias("seller_id"),   # safe here: only 1 distinct seller per order after filter
    )
    .filter(F.col("seller_count") == 1)
    .select("order_id", "seller_id")
)

# --- avg_delivery_days + pct_delayed_orders: ORDER grain, single-seller orders only ---
single_seller_order_metrics = (
    single_seller_orders
    .join(
        staging_orders.select(
            "order_id",
            "order_purchase_timestamp",
            "order_delivered_customer_date",
            "delivery_status",
        ),
        on="order_id",
        how="left",
    )
    .withColumn(
        "delivery_days",
        F.datediff(F.to_date("order_delivered_customer_date"), F.to_date("order_purchase_timestamp"))
    )
    .groupBy("seller_id")
    .agg(
        F.avg("delivery_days").alias("avg_delivery_days_single_seller"),
        F.count(F.when(F.col("delivery_status").isNotNull(), 1)).alias("completed_orders_single_seller"),
        F.count(F.when(F.col("delivery_status") == "delayed", 1)).alias("delayed_orders_single_seller"),
    )
    .withColumn(
        "pct_delayed_orders",
        F.when(
            F.col("completed_orders_single_seller") > 0,
            (F.col("delayed_orders_single_seller") / F.col("completed_orders_single_seller")).cast("decimal(5,4)")
        )
    )
)

# --- pct_shipped_late_single_seller: ITEM grain, single-seller orders only ---
shipping_punctuality = (
    staging_order_items
    .join(single_seller_orders.select("order_id"), on="order_id", how="inner")
    .join(
        staging_orders.select("order_id", "order_delivered_carrier_date"),
        on="order_id",
        how="left",
    )
    .filter(F.col("order_delivered_carrier_date").isNotNull())
    .withColumn(
        "shipped_late",
        F.when(F.col("order_delivered_carrier_date") > F.col("shipping_limit_date"), 1).otherwise(0)
    )
    .groupBy("seller_id")
    .agg(
        F.count("order_item_id").alias("single_seller_shipped_items"),
        F.sum("shipped_late").alias("shipped_late_items"),
    )
    .withColumn(
        "pct_shipped_late_single_seller",
        (F.col("shipped_late_items") / F.col("single_seller_shipped_items")).cast("decimal(5,4)")
    )
)

# --- overall revenue (all orders, regardless of seller count) ---
overall_seller_metrics = (
    staging_order_items
    .groupBy("seller_id")
    .agg(
        F.sum("price").cast("decimal(10,2)").alias("total_revenue"),
        F.countDistinct("order_id").alias("total_orders"),
        F.avg("freight_value").cast("decimal(10,2)").alias("avg_freight_value"),
    )
)

# --- combine all three pieces, keep only valid sellers ---
gold_revenue_by_seller = (
    overall_seller_metrics
    .join(single_seller_order_metrics, on="seller_id", how="left")
    .join(shipping_punctuality, on="seller_id", how="left")
    .join(staging_sellers.select("seller_id"), on="seller_id", how="inner")
    .orderBy(F.desc("total_revenue"))
)

full_table_name = "brazilian_ecommerce.prod.gold_revenue_by_seller"

(
    gold_revenue_by_seller.write
    .mode("overwrite")
    .option("comment", "gold table: revenue by seller (all orders); delivery/delay metrics and shipping punctuality restricted to single-seller orders")
    .option("overwriteSchema", "true")
    .saveAsTable(full_table_name)
)

spark.sql(f"""
    ALTER TABLE {full_table_name}
    ALTER COLUMN seller_id SET NOT NULL
""")

spark.sql(f"""
    ALTER TABLE {full_table_name}
    ADD CONSTRAINT pk_gold_revenue_by_seller PRIMARY KEY (seller_id)
""")

print(f"Loaded {gold_revenue_by_seller.count()} rows into {full_table_name}")